!!! This file should only be used if the entire database has been deleted. This will restore to the last 5 days of readings of the Richmond gauge. 

This file should only be used in event of not being able to restore the original database. 

In [ ]:
import requests
from datetime import datetime, timedelta, timezone
import psycopg
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

In [ ]:
# ---- CONFIG ----
measure_id = "0009-level-tidal_level-i-15_min-mAOD"

# ---- FETCH DATA FROM EA API ----
end = datetime.now(timezone.utc)
start = end - timedelta(days=5)

url = f"https://environment.data.gov.uk/flood-monitoring/id/measures/{measure_id}/readings"

params = {
    "startdate": start.strftime("%Y-%m-%d"),
    "enddate": end.strftime("%Y-%m-%d"),
    "_limit": 10000
}

r = requests.get(url, params=params)
r.raise_for_status()
data = r.json()["items"]

# ---- CONNECT TO POSTGRES (NEON) ----
with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        
        # Create table if not exists
        cur.execute("""
        CREATE TABLE IF NOT EXISTS richmond_tidal_levels (
            ts TIMESTAMPTZ PRIMARY KEY,
            water_level DOUBLE PRECISION NOT NULL,
            station_id TEXT NOT NULL,
            created_at TIMESTAMPTZ DEFAULT NOW()
        );
        """)
        
        # Insert rows safely (skip duplicates)
        for reading in data:
            cur.execute("""
                INSERT INTO richmond_tidal_levels (ts, water_level, station_id)
                VALUES (%s, %s, %s)
                ON CONFLICT (ts) DO NOTHING;
            """, (
                reading["dateTime"],
                reading["value"],
                "0009"
            ))

    conn.commit()

print(f"Inserted {len(data)} readings.")